# 第48章 统计折线图（lineplot）

用lineplot对重复观察进行统计聚合并显示时间趋势和误差。

## 学习目标

本章围绕一种明确的图表结构展开，先看最小可用示例，再加入分组、注释或交互细节。


## 适用场景

时间或有序X轴上，每个位置存在多条观察，需要展示均值与不确定性。

## 数据结构

长表，一列有序X、一列数值Y，可增加分组列。

## 本章练习任务

运行基础图表后，完成以下任务：

1. 将 errorbar=None 改为 errorbar="sd" 或 errorbar=("ci", 95)，观察误差区间的显示
2. 修改 estimator 为 "median"，对比均值线与中位数线的趋势差异
3. 添加 markers=False 参数，说明标记点对时间序列可读性的作用


## 0. 准备可复现数据

先完成导入和数据准备，后续单元格只负责一种图表或一种分析动作。


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

rng = np.random.default_rng(36)
n = 240
orders = pd.DataFrame({
    "category": rng.choice(["办公", "数码", "家居"], n, p=[0.34, 0.38, 0.28]),
    "channel": rng.choice(["自然流量", "广告", "会员"], n, p=[0.42, 0.36, 0.22]),
    "region": rng.choice(["华东", "华南", "华北"], n),
    "order_value": np.clip(rng.normal(260, 72, n), 45, None),
    "items": rng.integers(1, 7, n),
})
orders.loc[orders["category"] == "数码", "order_value"] *= 1.35
orders["satisfied"] = rng.choice(["满意", "一般"], n, p=[0.78, 0.22])

marketing = pd.DataFrame({
    "channel": rng.choice(["搜索", "社交", "会员"], n),
    "visits": rng.integers(80, 850, n),
    "ad_spend": rng.uniform(2, 38, n),
})
marketing["sales"] = (
    45 + marketing["visits"] * 0.16 + marketing["ad_spend"] * 2.4
    + marketing["channel"].map({"搜索": 18, "社交": 8, "会员": 32})
    + rng.normal(0, 28, n)
).clip(10)
marketing["conversion"] = (marketing["sales"] / marketing["visits"]).clip(0.02, 0.5)

daily = pd.DataFrame({
    "date": np.tile(pd.date_range("2026-01-01", periods=12, freq="D"), 3),
    "region": np.repeat(["华东", "华南", "华北"], 12),
})
daily["sales"] = (
    np.tile(np.linspace(110, 190, 12), 3)
    + np.repeat([28, 8, 18], 12)
    + rng.normal(0, 9, 36)
)

sns.set_theme(style="whitegrid", context="notebook")
print("订单样本:", orders.shape, "营销样本:", marketing.shape)


## 1. 基础图表

先保留必要的编码：位置、颜色或大小。图表标题、坐标轴和单位应能让读者脱离代码理解结果。


In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.3))
sns.lineplot(data=daily, x="date", y="sales", errorbar=None, color="#1a73e8", marker="o", ax=ax)
ax.set(title="每日平均销售额", xlabel="日期", ylabel="销售额")
ax.tick_params(axis="x", rotation=30)
fig.tight_layout()
plt.show()


## 2. 进阶变体

在基础图表可读的前提下增加分组、布局、注释或交互。新增编码必须服务于一个明确问题。


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
sns.lineplot(data=daily, x="date", y="sales", hue="region", marker="o", errorbar=None, palette="colorblind", ax=ax)
ax.set(title="区域每日销售趋势", xlabel="日期", ylabel="销售额")
ax.legend(title="区域", frameon=False)
ax.tick_params(axis="x", rotation=30)
fig.tight_layout()
plt.show()


## 3. 参数说明

- estimator：聚合函数
- errorbar：误差
- units：个体线
- sort：排序


## 4. 结果解读

默认线是各X位置的均值，阴影是误差区间；先确认聚合口径。


## 常见误区

- 把聚合线当成单个真实序列
- 日期未排序
- 误差阴影含义不清


## 综合练习

请使用同一份数据完成下面任务，并说明你选择该图表的原因。


In [ ]:
weekly = daily.copy()
weekly["day"] = weekly["date"].dt.day
fig, ax = plt.subplots(figsize=(8.5, 4.3))
sns.lineplot(data=weekly, x="day", y="sales", hue="region", style="region", markers=True, dashes=False, errorbar=None, ax=ax)
ax.set(title="按日序号比较区域趋势", xlabel="日", ylabel="销售额")
ax.legend(frameon=False)
fig.tight_layout()
plt.show()


## 本章小结

用lineplot对重复观察进行统计聚合并显示时间趋势和误差。


### 你已经掌握

- 判断统计折线图（lineplot）的适用场景
- 准备与图表匹配的数据结构
- 从基础图表扩展到分组、注释或交互变体
- 按照业务问题解读图表并说明结论边界


### 图表选择速查

| 选择要点 | 本章说明 |
| --- | --- |
| 适用场景 | 时间或有序X轴上，每个位置存在多条观察，需要展示均值与不确定性。 |
| 数据结构 | 长表，一列有序X、一列数值Y，可增加分组列。 |
| 结果解读 | 默认线是各X位置的均值，阴影是误差区间；先确认聚合口径。 |


### 关键参数

| 参数 | 作用 |
| --- | --- |
| `estimator` | 聚合函数 |
| `errorbar` | 误差 |
| `units` | 个体线 |
| `sort` | 排序 |


### 需要注意

- 把聚合线当成单个真实序列
- 日期未排序
- 误差阴影含义不清


### 完成检查

- [ ] 能判断什么问题适合使用统计折线图（lineplot）
- [ ] 能准备符合要求的数据结构
- [ ] 能独立完成基础图表和一个进阶变体
- [ ] 能调整关键参数并解释视觉变化
- [ ] 能根据图表写出有边界的数据结论
